In [1]:
from itertools import combinations_with_replacement
from functools import reduce
from more_itertools import flatten
import sympy as sp
from IPython.core.debugger import set_trace
from bidict import bidict
from typing import Any
sp.init_printing(use_latex=True)
from sympy import solve

In [2]:
x = sp.Symbol('x')
t = sp.Symbol('t')
u = sp.Function('u')(t, x)
heq = sp.Derivative(u, t) - sp.Derivative(u, x, x)

In [3]:
x = sp.Symbol('x')
t = sp.Symbol('t')
u = sp.Function('u')(t, x)
heq = sp.Derivative(u, t) - sp.Derivative(u, x, x)

def finish_substitution(expr):
    subs = set(expr.atoms(sp.Subs))
    subs_dic = {}
    for s0 in subs:
        bound = s0.bound_symbols
        der = s0.args[0]
        var = s0.args[2]
        subs_dic[s0] = der.xreplace(dict(zip(bound, var)))    
    return subs_dic

def ltf(expr, deps, indeps):
    """Lie Traditional Form."""
    #set_trace()
    functions = expr.atoms(sp.Function)
    reps = {}
    for fun in [_ for _ in functions if _ not in deps]:
        # Consider the case that some functions won't have the name
        # attribute e.g. Abs of an elementary function
        try:            
            reps[fun] = sp.Symbol(fun.name) # Otherwise functions with greek symbols aren't replaced
        except AttributeError:
            continue
    # first, resolve the dangling substitutions. Don't know why the
    # substitution is not done, but it seems that it has to do with
    # that a bound variable is within a function which is used as
    # a derivation argument
    subs_dic = finish_substitution(expr)
    output = expr
    output = output.xreplace(subs_dic)
    used_symbols = {}
    for deriv in output.atoms(sp.Derivative):
        # there is room to improve: collect indices and sort
        subindex = []
        for func_or_symbol, count in deriv.args[1:]:
            if func_or_symbol.is_Function:
                subindex.extend([func_or_symbol.name] * count)  
            elif func_or_symbol.is_Symbol:
                subindex.extend([f"{func_or_symbol}"] * count)
            else:
                print(func_or_symbol.__class__)
                display(output)
                print(func_or_symbol)
                a/a
        subindex = "{" + "".join(sorted(subindex)) + "}"
            
        fluffi = f"{deriv.args[0].name}_{subindex}"
        if fluffi in used_symbols:
            output = output.xreplace({deriv: used_symbols[fluffi]})
        else:
            s = sp.Symbol(fluffi)
            used_symbols[fluffi] = s
            output = output.xreplace({deriv: used_symbols[fluffi]})

    dreps2 = {}
    
    if len(indeps) == 1:
        # the original dependent variables should be written with primes   
        dreps2 = dict([(deriv, (sp.Symbol(deriv.output.subs(reps) +  
                                ' '.join("'" * deriv.args[-1][1]))))  \
                 for deriv in output.atoms(sp.Derivative) if deriv.args[0] in deps])
    else:
        derivatives = [_ for _ in output.atoms(sp.Derivative) if _.args[0] in deps]
        for dev in derivatives:
            s = ""
            for arg in dev.args[1:]:
                #print(f{arg[0]=}, {type(arg[0])}")
                match type(arg[0]):
                    case sp.Function:
                        n = arg[0].name
                    case sp.Symbol:
                        n = str(arg[0])
                    case _:
                        raise ValueError(f"{arg[0]} is type {type(arg[0])}")
                s += n*arg[1]
            dreps2[dev] = sp.Symbol(f"{dev.args[0].name}_"  + "{" +f"{s}" + "}")#

    fundic = dict([(_, sp.Symbol(_.name)) for _ in deps]) 
    output = output.xreplace(dreps2)
    output = output.xreplace(fundic)
    display(output.expand().simplify())   

In [4]:
ltf(heq, [u], [t,x])

In [5]:
def variable_combinations(variables: list[sp.Symbol], order:int) -> list(tuple[sp.Symbol]):
    return reduce(
        lambda acc, i: acc + list(map(list, combinations_with_replacement(variables, i))),
        range(1, order + 1),
        [])

In [6]:
infinitesimals = {}

In [7]:
def make_infinitesimal(v, *variables, name=""):
    return sp.Function(f'{v.name.swapcase() if not name else name}')(*variables)

In [8]:
def order(expr, deps, indeps):
    max_order = 0    
    max_deriv = set()
    k = expr.expand().atoms(sp.Derivative)
    for atom in k:
        if atom.args[0].name in [_.name for _ in deps]:
            _order = sum(cnt[1] for cnt in atom.args[1:])
            if max_order == _order:
                max_deriv |= set([atom])
            elif max_order < _order:
                max_deriv = set([atom])
                max_order = _order
    # XXX: return coefficients, too. Wehn some coefficients are -1, or 1, or numerical
    # return only those derivs
    return (max_order, max_deriv)

In [9]:
infinitesimals[x] = make_infinitesimal(x, t, x, u, name = "X")
infinitesimals[t] = make_infinitesimal(t, t, x, u, name = "T")
infinitesimals[u] = make_infinitesimal(u, t, x, u, name = "U")

In [10]:
_order = 2
var_combinatorics = variable_combinations([t, x], _order)

independents = [t, x]
dependents = [u]

def func_diff(fun:sp.Function, var:sp.Symbol | sp.Function) -> sp.Derivative:
    if var.is_Function or var.is_Derivative:
        d = sp.Symbol('d')
        r = fun.xreplace({var: d}).diff(d).xreplace({d: var}).doit()
    else:
        r = sp.Derivative(fun, var).doit()
    return r


def compute_level(deriv_vars_order: list[Any], dep, indep, infinitesimals):
    """Compute all derivatives and infinitesimals for a given derivative order.
    Extended Gamma operator (Arrigo, eq 2.85, or Schwarz, eq. 5.10)
    """
    v = deriv_vars_order[-1]
    # Base case (first order)
    if len(deriv_vars_order) == 1:
        funcs = dep
        etas = [infinitesimals[f] for f in funcs]
    else:
        prev_order = deriv_vars_order[:-1]
        funcs, etas = compute_level(prev_order, dep, indep, infinitesimals)
    # Compute current derivatives and infinitesimals functionally
    results = [
                (
                    func_diff(func, v),
                    reduce(
                        lambda acc, var: acc - func_diff(func, var)
                        * func_diff(infinitesimals[var], v),
                        indep,
                        func_diff(eta, v)
                    )
                )
                for func, eta in zip(funcs, etas)
            ]
    # Split result into separate lists
    funcs_next, etas_next = zip(*results)
    return list(funcs_next), list(etas_next)

# Compute all levels for each combination of derivative orders
#for _ in var_combinatorics:
#    vv = compute_level(_, dependents, independents, infinitesimals)
#    infinitesimals[vv[0][0]] = vv[1][0]

In [11]:
def rewrite_diff_equation_with_infinitesimal(equation, infinitesimals):
    if equation.is_Add:
        args = equation.expand().args
    else:
        args = [equation]
    acc = 0
    for term in args:
        if term.is_Derivative:
            local_term = [term]
        else:
            local_term = term.args
        f = 1
        for factor in local_term:
            if factor in infinitesimals:
                f *= infinitesimals[factor]
            else:
                f *= factor
        acc += f
    return acc

In [12]:
def prolongation(expr, n, _infinitesimals, dep, indep):
    for inf in _infinitesimals:
        d = finish_substitution(_infinitesimals[inf])
        _infinitesimals[inf] = _infinitesimals[inf].xreplace(d)
    
    expr = rewrite_diff_equation_with_infinitesimal(expr, _infinitesimals)
    return expr 

In [13]:
r=prolongation(heq, 2, infinitesimals, [u], [t, x])
ltf(r, dependents, independents)

In [14]:
o=order(heq,[u], [x,t])

In [15]:
res=solve(heq, list(o[1])[0])[0]

In [16]:
flauschi = finish_substitution(r)
r = r.xreplace(flauschi)
gurri = r.xreplace({list(o[1])[0]: res}).doit()

In [17]:
def extract_coeffs(expr, deps, indeps):
    def analyze_power(factor):
        base = factor.as_base_exp()[0]
        if base.is_Derivative:    
            if base.args[0] in deps:
                return factor
        #if base.is_Function:
        #    if base in deps:
        #        return factor        
        return 1   
    args = expr.expand().args
    all_i_need = set()
    for term in args:
        local_term = term.args
        f = 1
        for factor in local_term:
            if factor.is_Pow:
                f *= analyze_power(factor)
            elif factor.is_Derivative:
                if factor.args[0] in deps:
                    f *= factor
            elif factor.is_Function:
                #if factor in deps:
                #    f *= factor
                pass
            elif factor.is_number:
                pass
            else:
                # XXx: explore with heateq
                pass
        if f != 1:
            all_i_need.add(f)
    return list(all_i_need)

In [18]:
ec = extract_coeffs(gurri, [u], [x,t])

In [19]:
def get_coeff_order(expr):
    acc = 0
    if expr.is_Pow:
        acc += expr.as_base_exp()[1]
    elif expr.is_Mul:
        for a in expr.args:
            if a.is_Pow:
                acc += a.as_base_exp()[1]
            else:
                acc += 1
    else:
        acc += 1
    return acc
        

In [20]:
for _ in ec: get_coeff_order(_)

In [21]:
ltf(gurri, dependents, independents)
ec.sort(key=get_coeff_order, reverse=True)

In [22]:
def compute_determining_equations(expr, coeffs):
    acc = []
    for _ in coeffs:
        r = sum([term/_ for term in expr.expand().args if term.has(_)])
        acc.append(r)
        expr -= r*_
    acc.append(expr)
    return acc

In [23]:
ff=compute_determining_equations(gurri, ec)

In [24]:
for _ in ff:
    ltf(_, [u], [x,t])

In [25]:
def compute_overdetermined_system_of_infinitesimals(eq, dep, indep, infinitesimals):
    eq_order, highest_term = order(eq, dep, indep)
    highest_term = list(highest_term)[0]
    combos = variable_combinations(indep, eq_order)

    for comb in combos:
        funcs, etas = compute_level(comb, dep, indep, infinitesimals)
        infinitesimals[funcs[0]] = etas[0]

    r = prolongation(eq, eq_order, infinitesimals, dep, indep)
    sol = sp.solve(eq, highest_term)[0]
    r = r.xreplace(finish_substitution(r))
    r = r.xreplace({highest_term: sol})
    coeffs = sorted(
        extract_coeffs(r, dep, indep),
        key=get_coeff_order,
        reverse=True,
    )
    for _ in coeffs:
        ltf(_, dep, indep)
    return compute_determining_equations(r, coeffs)

In [ ]:
def main():
    t, x = sp.symbols("t x")
    u = sp.Function("u")(t, x)
    heq = sp.Derivative(u, t) - sp.Derivative(u, x, x)

    independents = [t, x]
    dependents = [u]

    infinitesimals = {
        x: make_infinitesimal(x, t, x, u, name="X"),
        t: make_infinitesimal(t, t, x, u, name="T"),
        u: make_infinitesimal(u, t, x, u, name="U"),
    }
    result = compute_overdetermined_system_of_infinitesimals(
        eq=heq,
        dep=dependents,
        indep=independents,
        infinitesimals=infinitesimals,
    )

    for eq in result:
        ltf(eq, [u], [x, t])

    return result


def main2():
    y, x = sp.symbols("y x")
    u = sp.Function("u")(x, y)
    laplace_eq = sp.Derivative(u, y, y) + sp.Derivative(u, x, x)

    print("Laplace equation")
    
    independents = [y, x]
    dependents = [u]

    infinitesimals = {
        x: make_infinitesimal(x, x, y, u, name="X"),
        y: make_infinitesimal(y, x, y, u, name="Y"),
        u: make_infinitesimal(u, x, y, u, name="U"),
    }

    result = compute_overdetermined_system_of_infinitesimals(
        eq=laplace_eq,
        dep=dependents,
        indep=independents,
        infinitesimals=infinitesimals,
    )

    for eq in set(result):
         ltf(eq, dependents, independents)
    return result


def main3():
    # XXX debug !!!
    t, x = sp.symbols("t x")
    u = sp.Function("u")(x, t)
    burgers_eq = sp.Derivative(u, t) + u * sp.Derivative(u, x) - sp.Derivative(u, x, x)
    print("Burgers equation")
    independents = [t, x]
    dependents = [u]

    infinitesimals = {
        x: make_infinitesimal(x, x, t, u, name="X"),
        t: make_infinitesimal(t, x, t, u, name="T"),
        u: make_infinitesimal(u, x, t, u, name="U"),
    }

    result = compute_overdetermined_system_of_infinitesimals(
        eq=burgers_eq,
        dep=dependents,
        indep=independents,
        infinitesimals=infinitesimals,
    )

    for eq in set(result):
        ltf(eq, dependents, independents)

    return result

def main4():
    x = sp.symbols("x")
    y = sp.Function("y")(x)
    ode= sp.Derivative(y, x,x)

    print("Arrigo Example 2.17")
    
    independents = [x]
    dependents = [y]

    infinitesimals = {
        x: make_infinitesimal(x, x, y, name="X"),
        y: make_infinitesimal(y, x, y, name="Y"),

    }

    result = compute_overdetermined_system_of_infinitesimals(
        eq=ode,
        dep=dependents,
        indep=independents,
        infinitesimals=infinitesimals,
    )

    for eq in set(result):
        ltf(eq, dependents, independents)

    return result


def main5():
    # XXX Debug
    # arrigo Example 2.18
    x = sp.symbols("x")
    y = sp.Function("y")(x)
    ode = sp.Derivative(y, x, x) + y*sp.Derivative(y, x) + x * y**4

    print("Arrigo Eaxmple 2.18")
    
    independents = [x]
    dependents = [y]

    infinitesimals = {
        x: make_infinitesimal(x, x, y, name="X"),
        y: make_infinitesimal(y, x, y, name="Y"),

    }

    result = compute_overdetermined_system_of_infinitesimals(
        eq=ode,
        dep=dependents,
        indep=independents,
        infinitesimals=infinitesimals,
    )

    for eq in set(result):
        ltf(eq, dependents, independents)

    return result


def main6():
    # XXX Debug
    print("arrigo Example 2.19")
    x = sp.symbols("x")
    y = sp.Function("y")(x)
    ode = sp.Derivative(y, x, x) + 3*y*sp.Derivative(y, x) + y**3

    independents = [x]
    dependents = [y]

    infinitesimals = {
        x: make_infinitesimal(x, x, y, name="X"),
        y: make_infinitesimal(y, x, y, name="Y"),

    }

    result = compute_overdetermined_system_of_infinitesimals(
        eq=ode,
        dep=dependents,
        indep=independents,
        infinitesimals=infinitesimals,
    )

    for eq in set(result):
        ltf(eq, dependents, independents)

    return result

def main7():
    print("Arrigo Example 2.20")
    x = sp.symbols("x")
    y = sp.Function("y")(x)
    ode = sp.Derivative(y, x, x, x) + y*sp.Derivative(y, x, x)

    independents = [x]
    dependents = [y]

    infinitesimals = {
        x: make_infinitesimal(x, x, y, name="X"),
        y: make_infinitesimal(y, x, y, name="Y"),

    }

    result = compute_overdetermined_system_of_infinitesimals(
        eq=ode,
        dep=dependents,
        indep=independents,
        infinitesimals=infinitesimals,
    )

    for eq in set(result):
        ltf(eq, dependents, independents)

    return result

def main8():
    # XXX Debug
    print("Arrigo Example 3.1, pp. 75")
    x = sp.symbols("x")
    t = sp.symbols("t")
    u = sp.Function("u")(x, t)
    pde = sp.Derivative(u, t) - sp.Derivative(u, x)**2

    
    independents = [x, t]
    dependents = [u]

    infinitesimals = {
        x: make_infinitesimal(x, x, t, u, name="X"),
        t: make_infinitesimal(t, x, t, u, name="T"),
        u: make_infinitesimal(u, x, t, u, name='U')
    }
    set_trace()
    result = compute_overdetermined_system_of_infinitesimals(
        eq=pde,
        dep=dependents,
        indep=independents,
        infinitesimals=infinitesimals,
    )

    for eq in result:
        ltf(eq, dependents, independents)

    return result

def main9():
    x = sp.symbols("x")
    y = sp.Function("y")(x)
    
    pde = sp.Derivative(y, x,x,x)  + y*sp.Derivative(y, x, x)

    independents = [x]
    dependents = [y]

    infinitesimals = {
        x: make_infinitesimal(x, x, y, name="X"),
        y: make_infinitesimal(y, x, y, name='Y')
    }
    result = compute_overdetermined_system_of_infinitesimals(
        eq=pde,
        dep=dependents,
        indep=independents,
        infinitesimals=infinitesimals,
    )

    print("Blasius Equation")
    for eq in result:
        ltf(eq, dependents, independents)

    return result
if __name__ == "__main__":
    pass
    #main()
    print("." * 80)
    main2()
    print("." * 80)
    main3()
    print("." * 80)
    main4()
    print("." * 80)
    main5()
    print("." * 80)
    main6()
    print("." * 80)
    main7()
    print("." * 80)
    main8()
    print("." * 80)
    main9()


................................................................................
Laplace equation


................................................................................
Burgers equation


................................................................................
Arrigo Example 2.17


................................................................................
Arrigo Eaxmple 2.18


................................................................................
arrigo Example 2.19


...

{x: X(t, x, u(t, x)), t: T(t, x, u(t, x)), u(t, x): U(t, x, u(t, x))}


ipdb>  p _infinitesimals


{x: X(x, t, u(x, t)), t: T(x, t, u(x, t)), u(x, t): U(x, t, u(x, t)), Derivative(u(x, t), x): -(Derivative(T(x, t, u(x, t)), x) + Derivative(T(x, t, u(x, t)), u(x, t))*Derivative(u(x, t), x))*Derivative(u(x, t), t) - (Derivative(X(x, t, u(x, t)), x) + Derivative(X(x, t, u(x, t)), u(x, t))*Derivative(u(x, t), x))*Derivative(u(x, t), x) + Derivative(U(x, t, u(x, t)), x) + Derivative(U(x, t, u(x, t)), u(x, t))*Derivative(u(x, t), x), Derivative(u(x, t), t): -(Derivative(T(x, t, u(x, t)), t) + Derivative(T(x, t, u(x, t)), u(x, t))*Derivative(u(x, t), t))*Derivative(u(x, t), t) - (Derivative(X(x, t, u(x, t)), t) + Derivative(X(x, t, u(x, t)), u(x, t))*Derivative(u(x, t), t))*Derivative(u(x, t), x) + Derivative(U(x, t, u(x, t)), t) + Derivative(U(x, t, u(x, t)), u(x, t))*Derivative(u(x, t), t)}


ipdb>  for _ in _infinitesimals: ltf(_, dep, indep)


ipdb>  for _ in _infinitesimals: ltf(_infinitesimals[_], dep, indep)
